In [ ]:
import pandas as pd
import numpy as np
import torch

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
RESULTS_PATH = "Phi3_Responces.csv"

df = pd.read_csv(RESULTS_PATH)

print("Number of questions:", len(df))
print("\nColumns:")
print(df.columns.tolist())

df.head()

In [ ]:
print("Missing values:")
print(df.isnull().sum())

In [ ]:
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)
print("Loading embedding model...")

embedder = SentenceTransformer(
    EMBEDDING_MODEL,
    device=device
)

print("Embedding model loaded.")

In [ ]:
def split_answers(text):
    if pd.isna(text):
        return []

    return [answer.strip() for answer in text.split(";") if answer.strip()]


df["Correct Answer List"] = df["Correct Answers"].apply(split_answers)
df["Incorrect Answer List"] = df["Incorrect Answers"].apply(split_answers)

print("Example correct answers:")
print(df.loc[0, "Correct Answer List"])

print("\nExample incorrect answers:")
print(df.loc[0, "Incorrect Answer List"])

In [ ]:
correct_texts = []
incorrect_texts = []

for answers in df["Correct Answer List"]:
    correct_texts.extend(answers)

for answers in df["Incorrect Answer List"]:
    incorrect_texts.extend(answers)

print("Total correct answers:", len(correct_texts))
print("Total incorrect answers:", len(incorrect_texts))

In [ ]:
print("Generating Phi-3 response embeddings...")

response_embeddings = embedder.encode(
    df["Phi3 Response"].tolist(),
    convert_to_numpy=True,
    show_progress_bar=True
)

print("Embedding shape:", response_embeddings.shape)

In [ ]:
print("Generating correct-answer embeddings...")

correct_embeddings = embedder.encode(
    correct_texts,
    convert_to_numpy=True,
    show_progress_bar=True
)

print("Generating incorrect-answer embeddings...")

incorrect_embeddings = embedder.encode(
    incorrect_texts,
    convert_to_numpy=True,
    show_progress_bar=True
)

print("\nCorrect embeddings shape:", correct_embeddings.shape)
print("Incorrect embeddings shape:", incorrect_embeddings.shape)

In [ ]:
# Calculate similarity between each Phi-3 response and its corresponding correct/incorrect reference answers
# Giving the model's prediction

results = []

correct_idx = 0
incorrect_idx = 0

for i in range(len(df)):

    response_embedding = response_embeddings[i].reshape(1, -1)

    n_correct = len(df.loc[i, "Correct Answer List"])
    n_incorrect = len(df.loc[i, "Incorrect Answer List"])

    question_correct_embeddings = correct_embeddings[
        correct_idx : correct_idx + n_correct
    ]

    question_incorrect_embeddings = incorrect_embeddings[
        incorrect_idx : incorrect_idx + n_incorrect
    ]

    correct_scores = cosine_similarity(
        response_embedding,
        question_correct_embeddings
    )[0]

    incorrect_scores = cosine_similarity(
        response_embedding,
        question_incorrect_embeddings
    )[0]

    max_correct_similarity = np.max(correct_scores)
    max_incorrect_similarity = np.max(incorrect_scores)

    # Detector prediction
    predicted_label = (
        "Correct"
        if max_correct_similarity > max_incorrect_similarity
        else "Hallucinated"
    )

    results.append({
        "max_correct_similarity": max_correct_similarity,
        "max_incorrect_similarity": max_incorrect_similarity,
        "Predicted_Label": predicted_label
    })

    correct_idx += n_correct
    incorrect_idx += n_incorrect


similarity_df = pd.DataFrame(results)

print("Similarity results shape:", similarity_df.shape)

similarity_df.head(5)

In [ ]:
# Combine the detector predictions with the original data
# Evaluation dataframe
evaluation_df = pd.concat(
    [
        df[
            [
                "Question",
                "Best Answer",
                "Correct Answers",
                "Incorrect Answers",
                "Phi3 Response"
            ]
        ].reset_index(drop=True),

        similarity_df.reset_index(drop=True)
    ],
    axis=1
)

print("Evaluation shape:", evaluation_df.shape)

evaluation_df.head(10)

In [ ]:
# Prepare prompts for an independent LLM evaluator

def create_evaluation_prompt(row):
    return f"""{row['Correct Answers']}

{row['Incorrect Answers']}

{row['Phi3 Response']}

Determine whether the model response correctly answers the question based on the reference answers.

Return ONLY one label:
Correct
or
Hallucinated
"""

evaluation_df["Evaluation_Prompt"] = evaluation_df.apply(
    create_evaluation_prompt,
    axis=1
)

evaluation_df[
    ["Question", "Phi3 Response", "Evaluation_Prompt"]
].head(3)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

EVALUATOR_MODEL = "Qwen/Qwen2.5-3B-Instruct"

print("Loading independent evaluator...")

evaluator_tokenizer = AutoTokenizer.from_pretrained(EVALUATOR_MODEL)

evaluator_model = AutoModelForCausalLM.from_pretrained(
    EVALUATOR_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

print("Evaluator loaded.")

In [ ]:
def evaluate_response(prompt):
    messages = [
        {
            "role": "system",
            "content": (
                "You are an independent factuality evaluator. "
                "Determine whether the model response correctly answers the question "
                "using the provided correct and incorrect reference answers. "
                "Return ONLY one label: Correct or Hallucinated."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = evaluator_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = evaluator_tokenizer(
        text,
        return_tensors="pt"
    ).to(evaluator_model.device)

    with torch.no_grad():
        outputs = evaluator_model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = evaluator_tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return response


print("Evaluating responses...")

llm_outputs = []

for i, prompt in enumerate(evaluation_df["Evaluation_Prompt"]):

    result = evaluate_response(prompt)

    llm_outputs.append(result)

    if (i + 1) % 20 == 0:
        print(f"Evaluated {i + 1}/200")

print("Evaluation complete.")

In [ ]:
def clean_label(output):
    output = output.strip().lower()

    if "hallucinated" in output:
        return "Hallucinated"

    if "correct" in output:
        return "Correct"

    return "Unclear"


evaluation_df["Evaluator_Output"] = llm_outputs

evaluation_df["True_Label"] = evaluation_df["Evaluator_Output"].apply(
    clean_label
)

print("True label counts:")
print(evaluation_df["True_Label"].value_counts())

In [ ]:
print("Unclear evaluations:")

unclear_df = evaluation_df[
    evaluation_df["True_Label"] == "Unclear"
]

print("Number of unclear labels:", len(unclear_df))

unclear_df[
    ["Question", "Phi3 Response", "Evaluator_Output"]
].head(20)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Remove any unclear evaluator results
final_eval_df = evaluation_df[
    evaluation_df["True_Label"].isin(["Correct", "Hallucinated"])
].copy()

# Independent ground truth
y_true = final_eval_df["True_Label"]

# Our embedding-based detector prediction
y_pred = final_eval_df["Predicted_Label"]

# Calculate metrics
accuracy = accuracy_score(y_true, y_pred)

precision = precision_score(
    y_true,
    y_pred,
    pos_label="Hallucinated",
    zero_division=0
)

recall = recall_score(
    y_true,
    y_pred,
    pos_label="Hallucinated",
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    pos_label="Hallucinated",
    zero_division=0
)

print(f"Number evaluated: {len(final_eval_df)}")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_true,
        y_pred,
        labels=["Correct", "Hallucinated"]
    )
)

In [ ]:
# Find cases where our detector disagrees with the independent evaluator

disagreements = final_eval_df[
    final_eval_df["True_Label"] != final_eval_df["Predicted_Label"]
].copy()

print("Number of disagreements:", len(disagreements))

disagreements[
    [
        "Question",
        "Phi3 Response",
        "Predicted_Label",
        "True_Label",
        "max_correct_similarity",
        "max_incorrect_similarity"
    ]
]

In [ ]:
print("Independent Ground Truth:")
print(y_true.value_counts())

print("\nDetector Predictions:")
print(y_pred.value_counts())

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

y_true = evaluation_df["True_Label"]
y_pred = evaluation_df["Predicted_Label"]

print("Baseline Results")
print("----------------")

print(f"Accuracy:  {accuracy_score(y_true, y_pred):.4f}")

print(f"Precision: {precision_score(
    y_true,
    y_pred,
    pos_label="Hallucinated",
    zero_division=0
):.4f}")

print(f"Recall:    {recall_score(
    y_true,
    y_pred,
    pos_label="Hallucinated",
    zero_division=0
):.4f}")

print(f"F1 Score:  {f1_score(
    y_true,
    y_pred,
    pos_label="Hallucinated",
    zero_division=0
):.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_true,
    y_pred,
    labels=["Correct", "Hallucinated"]
))

In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Similarity margin:
# positive = response is closer to a correct reference
# negative = response is closer to an incorrect reference

evaluation_df["Similarity_Margin"] = (
    evaluation_df["max_correct_similarity"]
    - evaluation_df["max_incorrect_similarity"]
)

# Independent ground truth
y_true = (evaluation_df["True_Label"] == "Correct").astype(int)

threshold_results = []

# Test a range of thresholds around the observed margins
thresholds = np.arange(-0.20, 0.21, 0.01)

for threshold in thresholds:

    # Predict Correct only when the correct-reference
    # similarity advantage is greater than the threshold
    y_pred = (
        evaluation_df["Similarity_Margin"] > threshold
    ).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(
            y_true, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_true, y_pred, zero_division=0
        ),
        "F1": f1_score(
            y_true, y_pred, zero_division=0
        )
    })

threshold_df = pd.DataFrame(threshold_results)

# Sort by F1
threshold_df = threshold_df.sort_values(
    "F1",
    ascending=False
)

threshold_df.head(15)

In [ ]:
# Analyze similarity margins by the independent true label

evaluation_df["Similarity_Margin"] = (
    evaluation_df["max_correct_similarity"]
    - evaluation_df["max_incorrect_similarity"]
)

print("Similarity margin statistics")
print("=" * 50)

print(
    evaluation_df.groupby("True_Label")["Similarity_Margin"]
    .describe()
)

print("\nMean margin by class:")
print(
    evaluation_df.groupby("True_Label")["Similarity_Margin"]
    .mean()
)

print("\nMedian margin by class:")
print(
    evaluation_df.groupby("True_Label")["Similarity_Margin"]
    .median()
)

In [ ]:
# Inspect the independently labeled Correct responses

correct_cases = evaluation_df[
    evaluation_df["True_Label"] == "Correct"
][[
    "Question",
    "Phi3 Response",
    "max_correct_similarity",
    "max_incorrect_similarity",
    "Similarity_Margin",
    "Predicted_Label"
]].sort_values(
    "Similarity_Margin",
    ascending=False
)

print("Independent Correct responses:", len(correct_cases))

correct_cases

## Analysis of Current Reference-Similarity Detector

The current reference-similarity detector does not provide sufficient separation to serve as the final hallucination detector.

### Key Findings

- Correct responses have a higher mean similarity margin (**0.0615**) than hallucinated responses (**−0.0092**).
- However, the median margins are very close (**0.0114 vs. 0.0036**), indicating substantial overlap.
- **3 of 8** independently labeled Correct responses are misclassified as Hallucinated.
- The evaluation set is highly imbalanced (**8 Correct vs. 192 Hallucinated**).
- Therefore, optimizing a threshold at this stage may overfit the current data rather than improve general detection performance.

### Next Step

Instead of relying only on:

**Maximum Correct Similarity − Maximum Incorrect Similarity**

we will evaluate a richer feature set:

1. Maximum similarity to correct references
2. Maximum similarity to incorrect references
3. Difference between maximum similarities
4. Average similarity to correct references
5. Average similarity to incorrect references

Using these features should provide more information about the overall semantic relationship between each Phi-3 response and its reference answers.

In [ ]:
# Calculate average similarity to correct and incorrect references
# for each Phi-3 response

avg_results = []

correct_idx = 0
incorrect_idx = 0

for i in range(len(df)):

    response_embedding = response_embeddings[i].reshape(1, -1)

    n_correct = len(df.loc[i, "Correct Answer List"])
    n_incorrect = len(df.loc[i, "Incorrect Answer List"])

    question_correct_embeddings = correct_embeddings[
        correct_idx : correct_idx + n_correct
    ]

    question_incorrect_embeddings = incorrect_embeddings[
        incorrect_idx : incorrect_idx + n_incorrect
    ]

    # Similarity to all correct references
    correct_scores = cosine_similarity(
        response_embedding,
        question_correct_embeddings
    )[0]

    # Similarity to all incorrect references
    incorrect_scores = cosine_similarity(
        response_embedding,
        question_incorrect_embeddings
    )[0]

    avg_results.append({
        "avg_correct_similarity": np.mean(correct_scores),
        "avg_incorrect_similarity": np.mean(incorrect_scores),
        "max_correct_similarity": np.max(correct_scores),
        "max_incorrect_similarity": np.max(incorrect_scores)
    })

    correct_idx += n_correct
    incorrect_idx += n_incorrect

avg_similarity_df = pd.DataFrame(avg_results)

# Add the new features to evaluation_df
evaluation_df["avg_correct_similarity"] = (
    avg_similarity_df["avg_correct_similarity"].values
)

evaluation_df["avg_incorrect_similarity"] = (
    avg_similarity_df["avg_incorrect_similarity"].values
)

# Average similarity margin
evaluation_df["avg_similarity_margin"] = (
    evaluation_df["avg_correct_similarity"]
    - evaluation_df["avg_incorrect_similarity"]
)

print("New similarity features created.")

evaluation_df[
    [
        "Question",
        "True_Label",
        "max_correct_similarity",
        "max_incorrect_similarity",
        "Similarity_Margin",
        "avg_correct_similarity",
        "avg_incorrect_similarity",
        "avg_similarity_margin"
    ]
].head(10)

In [ ]:
# Test combined similarity features against independent ground truth

evaluation_df["Combined_Score"] = (
    0.5 * evaluation_df["Similarity_Margin"]
    + 0.5 * evaluation_df["avg_similarity_margin"]
)

y_true = (
    evaluation_df["True_Label"] == "Correct"
).astype(int)

# Test thresholds on the combined score

thresholds = np.arange(-0.20, 0.21, 0.01)

combined_results = []

for threshold in thresholds:

    y_pred = (
        evaluation_df["Combined_Score"] > threshold
    ).astype(int)

    combined_results.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(
            y_true, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_true, y_pred, zero_division=0
        ),
        "F1": f1_score(
            y_true, y_pred, zero_division=0
        )
    })

combined_threshold_df = (
    pd.DataFrame(combined_results)
    .sort_values("F1", ascending=False)
)

combined_threshold_df.head(10)

## Why We Need a Better Detector

The similarity-only approach is not a strong final classifier against the independent ground truth.

### Current Results

- Accuracy: **95.0%**
- Precision: **25.0%**
- Recall: **12.5%**
- F1 Score: **16.7%**

The high accuracy is misleading because the dataset is highly imbalanced (**192 Hallucinated vs. 8 Correct**). The low recall and F1 show that the detector struggles to identify the minority Correct class.

### Conclusion

We should stop threshold tuning.

> **Embedding similarity provides useful signals, but max/average similarity alone is not sufficient to reliably classify hallucinations in this dataset.**

This does **not** mean the project has failed. Instead, the independent LLM labels give us a legitimate supervised target for developing a better detector.

### Next Step: Feature-Based Classifier

We will use the similarity measurements as features for a classifier.

Current features:

- `max_correct_similarity`
- `max_incorrect_similarity`
- `Similarity_Margin`
- `avg_similarity_margin`

We can also include the raw average similarities to the correct and incorrect references.

Because there are only **8 Correct examples**, we should avoid training and evaluating on the same 200 samples. We will use **stratified cross-validation** so that each fold preserves the class distribution as closely as possible. :contentReference[oaicite:0]{index=0}

This provides a more defensible estimate of how well the feature-based detector generalizes.

In [ ]:
# Inspect all independently labelled Correct responses

correct_cases = evaluation_df[
    evaluation_df["True_Label"] == "Correct"
][[
    "Question",
    "Best Answer",
    "Correct Answers",
    "Phi3 Response",
    "True_Label",
    "Predicted_Label"
]]

print("Independent Correct responses:", len(correct_cases))

correct_cases

In [ ]:
# Analyze detector errors

errors_df = evaluation_df[
    evaluation_df["True_Label"] != evaluation_df["Predicted_Label"]
].copy()

print("Total errors:", len(errors_df))

print("\nFalse Positives:")
print(
    ((errors_df["True_Label"] == "Hallucinated") &
     (errors_df["Predicted_Label"] == "Correct")).sum()
)

print("\nFalse Negatives:")
print(
    ((errors_df["True_Label"] == "Correct") &
     (errors_df["Predicted_Label"] == "Hallucinated")).sum()
)

errors_df[
    [
        "Question",
        "Phi3 Response",
        "True_Label",
        "Predicted_Label",
        "max_correct_similarity",
        "max_incorrect_similarity",
        "Similarity_Margin"
    ]
].head(20)

In [ ]:
# Analyze false positives

false_positives = evaluation_df[
    (evaluation_df["True_Label"] == "Hallucinated") &
    (evaluation_df["Predicted_Label"] == "Correct")
].copy()

print("False Positives:", len(false_positives))

false_positives[
    [
        "Question",
        "Phi3 Response",
        "max_correct_similarity",
        "max_incorrect_similarity",
        "Similarity_Margin"
    ]
].sort_values(
    "Similarity_Margin",
    ascending=False
).head(20)

In [ ]:
# Balanced Logistic Regression with cross-validation

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

cv = StratifiedKFold(
    n_splits=4,
    shuffle=True,
    random_state=42
)

model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        random_state=42,
        max_iter=1000
    ))
])

y_pred_cv = cross_val_predict(
    model,
    X,
    y,
    cv=cv
)

print("Cross-validated results:")
print(f"Accuracy:  {accuracy_score(y, y_pred_cv):.4f}")
print(f"Precision: {precision_score(y, y_pred_cv, pos_label='Hallucinated'):.4f}")
print(f"Recall:    {recall_score(y, y_pred_cv, pos_label='Hallucinated'):.4f}")
print(f"F1 Score:  {f1_score(y, y_pred_cv, pos_label='Hallucinated'):.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(
    y,
    y_pred_cv,
    labels=["Correct", "Hallucinated"]
))

In [ ]:
# Store cross-validated predictions

evaluation_df["ML_Predicted_Label"] = y_pred_cv

# Compare baseline detector vs. Logistic Regression

print("Baseline similarity detector:")
print(f"Accuracy: {accuracy_score(y, evaluation_df['Predicted_Label']):.4f}")
print(f"Precision: {precision_score(y, evaluation_df['Predicted_Label'], pos_label='Hallucinated'):.4f}")
print(f"Recall: {recall_score(y, evaluation_df['Predicted_Label'], pos_label='Hallucinated'):.4f}")
print(f"F1: {f1_score(y, evaluation_df['Predicted_Label'], pos_label='Hallucinated'):.4f}")

print("\nLogistic Regression:")
print(f"Accuracy: {accuracy_score(y, y_pred_cv):.4f}")
print(f"Precision: {precision_score(y, y_pred_cv, pos_label='Hallucinated'):.4f}")
print(f"Recall: {recall_score(y, y_pred_cv, pos_label='Hallucinated'):.4f}")
print(f"F1: {f1_score(y, y_pred_cv, pos_label='Hallucinated'):.4f}")

In [ ]:
# Final confusion matrix for the improved model

from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

ConfusionMatrixDisplay.from_predictions(
    y,
    y_pred_cv,
    labels=["Correct", "Hallucinated"],
    display_labels=["Correct", "Hallucinated"],
    cmap="Blues"
)

plt.title("Logistic Regression Confusion Matrix")
plt.show()